# Практика 22. Часовий ряд: побудова, ковзне середнє, виділення тренду і прогноз

**Варіант 1: Київ**

Мета: побудувати 10-річний щомісячний ряд температури, виділити тренд і сезонність, порівняти способи згладжування та обережно спрогнозувати наступні 12 місяців.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

variant = 1
city = 'Київ'
base_temp = 9.5
amplitude = 14.0
trend_per_year = 0.04
noise_scale = 1.0
alpha = 0.05

np.random.seed(variant)
n_years = 10
dates = pd.date_range(start='2015-01-01', periods=n_years * 12, freq='MS')
month = dates.month
years_elapsed = (dates.year - dates.year[0]) + (dates.month - 1) / 12
seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
trend = trend_per_year * years_elapsed
noise = np.random.normal(0, noise_scale, len(dates))
temp = np.round(base_temp + trend + seasonal + noise, 1)
climate_ts = pd.Series(temp, index=dates, name='температура')

print(f'Варіант {variant}: {city}')
print(f'Період: {climate_ts.index[0].date()} — {climate_ts.index[-1].date()}')
print(f'Кількість спостережень: {len(climate_ts)}')
print(climate_ts.head(12))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(climate_ts.index, climate_ts, color='teal')
ax.set_xlabel('Дата')
ax.set_ylabel('Температура, °C')
ax.set_title('10-річний щомісячний ряд температури Києва')
ax.grid(alpha=0.3)
plt.show()

## Завдання 1. Побудова 10-річного часового ряду

Будуємо щомісячну температуру Києва за 2015–2024 роки. Очікуємо повільний підйом рівня через тренд `0.04 °C/рік` і повторюваний річний цикл.

In [ ]:
ma3 = climate_ts.rolling(window=3, center=True).mean()
ma12_centered = climate_ts.rolling(window=12, center=True).mean()
ma12_trailing = climate_ts.rolling(window=12).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(climate_ts.index, climate_ts, alpha=0.35, label='Оригінал')
ax.plot(ma3.index, ma3, label='Центроване вікно 3')
ax.plot(ma12_centered.index, ma12_centered, label='Центроване вікно 12')
ax.plot(ma12_trailing.index, ma12_trailing, linestyle='--', label='Trailing вікно 12')
ax.set_xlabel('Дата')
ax.set_ylabel('Температура, °C')
ax.set_title('Ковзні середні та вплив вікна')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print('Вікно 12 краще прибирає річну сезонність; trailing-крива запізнюється відносно центрованої.')

## Завдання 2. Ковзне середнє: згладжування й лаг

Порівнюємо центровані вікна 3 і 12 місяців із трейлінговим вікном 12 місяців. Вікно 12 відповідає річному сезонному циклу, а trailing-варіант використовує лише минулі значення.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import linregress

decomposition = seasonal_decompose(climate_ts, model='additive', period=12)
trend_clean = decomposition.trend.dropna()
seasonal_monthly = decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
resid_std = decomposition.resid.dropna().std()

print('Типовий сезонний профіль за місяцями:')
print(seasonal_monthly.round(3))
print(f'Найвищий сезонний ефект: місяць {seasonal_monthly.idxmax()}')
print(f'Найнижчий сезонний ефект: місяць {seasonal_monthly.idxmin()}')
print(f'Тренд: від {trend_clean.iloc[0]:.3f} до {trend_clean.iloc[-1]:.3f} °C')
print(f'Стандартне відхилення залишків: {resid_std:.3f} °C; заданий noise_scale = {noise_scale:.1f} °C')
fig = decomposition.plot()
fig.set_size_inches(11, 7)
plt.show()

## Завдання 3. Декомпозиція `seasonal_decompose`

Виконуємо адитивну декомпозицію з періодом 12. Інтерпретуємо тренд, типовий сезонний профіль і залишок окремо.

In [ ]:
n_ahead = 12
x_future = np.arange(len(trend_clean), len(trend_clean) + n_ahead)
trend_forecast = fit.intercept + fit.slope * x_future
seasonal_by_month = decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
future_dates = pd.date_range(climate_ts.index[-1] + pd.offsets.MonthBegin(1), periods=n_ahead, freq='MS')
seasonal_forecast = seasonal_by_month.reindex(future_dates.month).to_numpy()
forecast = pd.Series(trend_forecast + seasonal_forecast, index=future_dates, name='прогноз')

print(forecast.round(2))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(climate_ts.index, climate_ts, label='Історичні дані')
ax.plot(forecast.index, forecast, marker='o', color='crimson', label='Прогноз')
ax.set_xlabel('Дата')
ax.set_ylabel('Температура, °C')
ax.set_title('Прогноз температури на 12 місяців')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print('Застереження: прогноз не враховує можливі зміни клімату, режиму вимірювань або зовнішніх факторів; дальша екстраполяція менш надійна.')

## Завдання 4. Прогноз на 12 місяців уперед

Продовжуємо лінійний тренд і додаємо типовий сезонний ефект відповідного календарного місяця. Це екстраполяція: майбутні числа залежать від припущення, що нахил тренду та сезонний профіль не зміняться.

In [ ]:
slope_ci_low = fit.slope - 1.96 * fit.stderr
slope_ci_high = fit.slope + 1.96 * fit.stderr
short_start = len(climate_ts) - 36
short_fit = linregress(np.arange(36), climate_ts.iloc[short_start:].values)

print(f'Повний ряд: slope = {fit.slope:.6f} °C/місяць, p-value = {fit.pvalue:.6g}, stderr = {fit.stderr:.6f}')
print(f'95%-й ДІ для slope: ({slope_ci_low:.6f}, {slope_ci_high:.6f})')
print(f'Останні 3 роки: slope = {short_fit.slope:.6f} °C/місяць, p-value = {short_fit.pvalue:.6g}')
print('H₀ про нульовий тренд відхиляємо.' if fit.pvalue < alpha else 'H₀ про нульовий тренд не відхиляємо.')
print('Десятирічний ряд дає статистичні підстави говорити про тренд потепління.' if fit.pvalue < alpha and slope_ci_low > 0 else 'Десятирічний ряд не дає достатніх статистичних підстав говорити про потепління.')

## Завдання 5. Реальний тренд чи шум на короткому вікні?

Перевіряємо H₀: істинний нахил тренду дорівнює нулю, тобто видиме потепління може бути лише шумом. Порівнюємо p-value, 95%-й інтервал і нахили на всьому ряді та останніх трьох роках.

## Контрольні питання

**1. Чому часовий ряд не є звичайною незалежною вибіркою?** Сусідні спостереження можуть бути пов'язані в часі, а сезонність і тренд порушують однаковість розподілу. Порядок дат містить інформацію, якої немає в довільній перестановці даних.

**2. Чому вікно узгоджують із сезонністю?** Маленьке вікно залишає сезонний шум, а надто велике може змити короткі зміни й створити сильні крайові втрати. Вікно 12 відповідає річному циклу й краще прибирає сезонність.

**3. Чим відрізняються trailing і centered?** Trailing використовує поточне та минулі значення й придатне для реального часу, але запізнюється за трендом. Centered бере майбутні значення навколо центру, тому краще вирівнює ряд заднім числом, але недоступне в реальному часі.

**4. Чому важливе p-value нахилу?** Додатний slope може з'явитися випадково. Лише p-value та інтервал оцінюють, чи є достатні підстави відрізнити реальний тренд від шуму.

## Підсумок

- Побудовано 10-річний щомісячний ряд температури Києва з сезонністю, трендом і шумом.
- Порівняно ковзні середні для різних вікон і трейлінгове/центроване згладжування.
- Виконано декомпозицію на тренд, сезонність і залишок.
- Побудовано 12-місячний прогноз і статистично перевірено нахил тренду.